# 31 — Ensamble de modelos de texto (logística + BETO + modelos nuevos)

**Idea propia.** La Fase T mostró que lo que suma es **combinar modelos de texto que se equivocan distinto**: (logística + BETO)/2 en lugar de `prob_texto_new` llevó el promedio final a 0,9435 (sin historial 0,8962). Acá se suman modelos de texto más baratos y distintos. **Nueva referencia: variante a de la Fase T.** Todo con la semilla 42 (la de BETO guardado).

**Partes:**
1. Modelos nuevos (título + garantía), cada uno envuelto en `PrediccionOOF`:
   - (a) Naive Bayes Multinomial sobre conteos de palabras 1-2;
   - (b) Naive Bayes Bernoulli (presencia/ausencia de palabras 1-2);
   - (c) logística solo con n-gramas de caracteres 3-5;
   - (d) logística solo con palabras 1-2;
   - (e) SVM lineal sobre el TF-IDF de la logística actual, con `CalibratedClassifierCV` para tener probabilidades.
2. Combinaciones como UNA columna (promedio simple) en lugar de `prob_texto_new`: a + cada modelo nuevo, todos, y el subconjunto elegido con 4 folds (*log-loss*) y evaluado en el quinto.
3. HistGB P7a + duración y promedio final, comparación pareada **contra la variante a**. Accuracy total y en vendedores sin historial.

**Compatibilidad del cross-fitting.** Los modelos nuevos usan el **mismo** `PrediccionOOF` que la logística: en las filas de train del fold externo, OOF de una CV interna sobre el train del fold; en la validación, un modelo entrenado con todo el train del fold. BETO: `sin_j_k` / `sin_k` (Fase Q). Ninguna probabilidad de una fila de train sale de un modelo que la vio, y ningún modelo vio la validación: se pueden promediar. **Control:** la variante a recalculada tiene que reproducir exactamente las OOF guardadas de la Fase T.

**Material:**
- Naive Bayes Multinomial y Bernoulli: `material/Naive Bayes, LDA, QDA/Presentación_NB_LDA_QDA.pdf` (slides 11 y 27) y `Copia de AP_NaiveBayes.ipynb` (`MultinomialNB()` y `BernoulliNB()` con parámetros por defecto, como acá);
- regresión logística: `material/Regresion Logistica/Presentación_Logística.pdf`;
- SVM lineal: `material/SVM/SVM (1).pdf`, slide 52 (Linear SVC), parámetros por defecto;
- promedio de probabilidades (voting *soft*): SIA `S8 - Ensamble.pdf`, slides 6-7; *log-loss*: `Árboles.pdf`, slides 12-13.

**No está en el material:** `CalibratedClassifierCV` (*Platt scaling*). El SVM no da probabilidades sino una distancia a la frontera; `method="sigmoid"` ajusta una logística de una sola variable (esa distancia) con una CV interna de 5 folds, para convertirla en probabilidad sin usar las mismas filas con las que se entrenó el SVM. Hace falta para poder promediar. TF-IDF y n-gramas de caracteres: ver `03_titulo` / `09_texto`.

In [1]:
import sys
sys.path.append("..")
import os, time, warnings
from itertools import combinations
import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
                      PrediccionOOF, armar_modelo_texto, armar_pipeline_histgb_combos, armar_pipeline_final, INDICES_CAT)

pd.set_option("display.width", 220)
SEED = 42
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
y_np = y.to_numpy()
n = len(crudo)
train = agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(crudo))))
folds = np.empty(n, dtype=int)
for k, (_, v) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(train, y)):
    folds[v] = k + 1
carpeta_lab = ".cache_laboratorio"
carpeta_bert = os.path.join(carpeta_lab, "bert")
carpeta_textos = os.path.join(carpeta_lab, "textos_31")
os.makedirs(carpeta_textos, exist_ok=True)
memoria_texto = Memory(location=carpeta_lab, verbose=0)


def acc_por_fold(prob, filas=None):
    filas = np.ones(n, bool) if filas is None else filas
    acierto = ((np.asarray(prob) >= 0.5).astype(int) == y_np)
    return pd.Series(acierto[filas]).groupby(folds[filas]).mean().to_numpy()


def comparar(acc, ref):
    diferencias = np.round(acc - ref, 10)
    gana = int((diferencias > 0).sum())
    return {"media": acc.mean(), "dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
            "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())}


sin_historial = np.zeros(n, bool)
for k in range(1, 6):
    va = folds == k
    sin_historial[va] = ~train.loc[va, "seller_id"].isin(set(train.loc[~va, "seller_id"])).to_numpy()

# Referencia: variante a de la Fase T (OOF guardadas por la 29)
oof_t = pd.read_csv(os.path.join(carpeta_lab, "oof_texto_combinado.csv"))
assert (oof_t["fold"].to_numpy() == folds).all()
ref_hgb = oof_t["(a) promedio simple"].to_numpy()
ref_final = 0.5 * oof_t["catboost (a) promedio simple"].to_numpy() + 0.5 * ref_hgb
acc_ref_hgb, acc_ref_final = acc_por_fold(ref_hgb), acc_por_fold(ref_final)
print("Variante a — HistGB:", acc_ref_hgb.mean().round(4), "| promedio final:", acc_ref_final.mean().round(4),
      "| sin historial (final):", acc_por_fold(ref_final, sin_historial).mean().round(4))

# BETO (Fase Q)
bert_global = np.full(n, np.nan)
for k in range(1, 6):
    d = np.load(os.path.join(carpeta_bert, f"sin_{k}.npz"))
    es_train = d["filas"] < n
    bert_global[d["filas"][es_train]] = d["prob"][es_train]
bert_pares = {}
for j in range(1, 6):
    for k in range(j + 1, 6):
        d = np.load(os.path.join(carpeta_bert, f"sin_{j}_{k}.npz"))
        p = np.full(n, np.nan)
        p[d["filas"]] = d["prob"]
        bert_pares[(j, k)] = bert_pares[(k, j)] = p


def columna_bert(k):
    col = bert_global.copy()
    for j in range(1, 6):
        if j != k:
            col[folds == j] = bert_pares[(j, k)][folds == j]
    return col

Variante a — HistGB: 0.9425 | promedio final: 0.9435 | sin historial (final): 0.8962


## 1. Modelos de texto nuevos

**Qué hace `MODELOS_TEXTO`:** define los 5 modelos (más la logística actual como `logistica`). Todos leen `title` y `warranty_texto`.

**Qué hace `texto_fold(nombre, k)`:** envuelve el modelo en `PrediccionOOF(modelo, random_state=42)` (igual que el `Pipeline` del modelo final), hace `fit_transform` sobre el train del fold k (OOF interna) y `transform` sobre la validación. Devuelve un vector de 70.000 valores y lo guarda en `textos_31/` para no recalcularlo. La logística actual usa además la caché de siempre.

In [2]:
def palabras(binario=False):
    return CountVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode", binary=binario)


def tfidf(analizador):
    rango = (3, 5) if analizador == "char_wb" else (1, 2)
    return TfidfVectorizer(analyzer=analizador, ngram_range=rango, min_df=2, strip_accents="unicode")


def titulo_y_garantia(vectorizador):
    return ColumnTransformer([("titulo", clone(vectorizador), "title"), ("garantia", clone(vectorizador), "warranty_texto")])


MODELOS_TEXTO = {
    "nb_multinomial": Pipeline([("vectores", titulo_y_garantia(palabras())), ("modelo", MultinomialNB())]),
    "nb_bernoulli": Pipeline([("vectores", titulo_y_garantia(palabras(binario=True))), ("modelo", BernoulliNB())]),
    "log_caracteres": Pipeline([("vectores", titulo_y_garantia(tfidf("char_wb"))),
                                ("modelo", LogisticRegression(C=2, max_iter=1000, random_state=SEED))]),
    "log_palabras": Pipeline([("vectores", titulo_y_garantia(tfidf("word"))),
                              ("modelo", LogisticRegression(C=2, max_iter=1000, random_state=SEED))]),
    "svm_lineal": Pipeline([("vectores", armar_modelo_texto().named_steps["vectores"]),
                            ("modelo", CalibratedClassifierCV(LinearSVC(random_state=SEED), method="sigmoid", cv=5))]),
}
NUEVOS = list(MODELOS_TEXTO)
COLUMNAS = ["title", "warranty_texto"]


def texto_fold(nombre, k):
    archivo = os.path.join(carpeta_textos, f"{nombre}_fold{k}.npy")
    if os.path.exists(archivo):
        return nombre, k, np.load(archivo)
    entrena, valida = folds != k, folds == k
    inicio = time.time()
    if nombre == "logistica":
        oof = Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)), ("identidad", "passthrough")],
                       memory=memoria_texto)
    else:
        oof = PrediccionOOF(MODELOS_TEXTO[nombre], random_state=SEED)
    valores = np.empty(n)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        valores[entrena] = oof.fit_transform(train.loc[entrena, COLUMNAS], y[entrena])[:, 0]
        valores[valida] = oof.transform(train.loc[valida, COLUMNAS])[:, 0]
    np.save(archivo, valores)
    print(f"{nombre} fold {k}: {(time.time() - inicio) / 60:.1f} min", flush=True)
    return nombre, k, valores


inicio = time.time()
salida = Parallel(n_jobs=5)(delayed(texto_fold)(m, k) for m in ["logistica"] + NUEVOS for k in range(1, 6))
componentes = {k: {"beto": columna_bert(k)} for k in range(1, 6)}
for nombre, k, valores in salida:
    componentes[k][nombre] = valores
print(f"Modelos de texto: {(time.time() - inicio) / 60:.1f} min")


def oof_externa(nombre):
    # Probabilidad de cada fila predicha por el modelo del fold externo que no la vio
    p = np.empty(n)
    for k in range(1, 6):
        p[folds == k] = componentes[k][nombre][folds == k]
    return p


externas = {m: oof_externa(m) for m in ["logistica", "beto"] + NUEVOS}
assert np.allclose(externas["logistica"], pd.read_csv(os.path.join(carpeta_lab, "oof_texto_logistica.csv"))["prob"])
assert np.allclose(externas["beto"], bert_global)

logistica fold 1: 0.0 min


logistica fold 2: 0.0 min


logistica fold 3: 0.0 min


logistica fold 4: 0.0 min
logistica fold 5: 0.0 min


nb_multinomial fold 1: 0.1 min


nb_multinomial fold 2: 0.1 min


... (20 outputs left out)

svm_lineal fold 3: 0.8 min


svm_lineal fold 4: 0.8 min


svm_lineal fold 5: 0.8 min


Modelos de texto: 2.0 min


### Cada modelo de texto solo

Accuracy (total y en vendedores sin historial), correlación con la logística y con BETO, y % de filas donde discrepan (uno dice `new` y el otro `used`). Un modelo sirve para el ensamble si acierta razonablemente y **se equivoca distinto** (correlación baja, discrepancia alta).

In [3]:
filas = {}
for m, p in externas.items():
    filas[m] = {"acc_total": acc_por_fold(p).mean(), "acc_sin_hist": acc_por_fold(p, sin_historial).mean(),
                "corr_logistica": np.corrcoef(p, externas["logistica"])[0, 1], "corr_beto": np.corrcoef(p, externas["beto"])[0, 1],
                "discrepa_logistica": ((p >= 0.5) != (externas["logistica"] >= 0.5)).mean(),
                "discrepa_beto": ((p >= 0.5) != (externas["beto"] >= 0.5)).mean()}
solos = pd.DataFrame(filas).T
solos.round(4)

,acc_total,acc_sin_hist,corr_logistica,corr_beto,discrepa_logistica,discrepa_beto
logistica,0.8414,0.7835,1.0000,0.8887,0.0000,0.1171
beto,0.8497,0.7926,0.8887,1.0000,0.1171,0.0000
nb_multinomial,0.8168,0.7397,0.8780,0.8092,0.1116,0.1463
nb_bernoulli,0.8055,0.7389,0.8638,0.8042,0.1354,0.1546
log_caracteres,0.8322,0.7733,0.9743,0.8819,0.0562,0.1195
log_palabras,0.8370,0.7753,0.9797,0.8803,0.0490,0.1220
svm_lineal,0.8351,0.7768,0.9859,0.8634,0.0408,0.1314


## 2. Combinaciones (una sola columna de texto)

**Qué hace `promedio(k, miembros)`:** promedio simple de las probabilidades de los miembros, para el fold externo k (filas de train: OOF internas; validación: modelos del train del fold).

Candidatas fijas:
- `a` (referencia): logística + BETO;
- `a + X` para cada modelo nuevo X;
- `todos`: los 7.

**Subconjunto elegido (*log-loss*, 4 folds → quinto):** para el fold k se prueban los 32 subconjuntos de los 5 modelos nuevos (siempre con logística + BETO) y se elige el de menor *log-loss* de la columna promediada sobre las **filas de train del fold** (los otros 4 folds), que es lo que ve el HistGB. Se aplica a la validación del fold k.

Tabla descriptiva: *log-loss* y accuracy de cada columna de texto sola (OOF externa).

In [4]:
def promedio(k, miembros):
    return np.mean([componentes[k][m] for m in miembros], axis=0)


def log_loss(p, y_):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return -np.mean(y_ * np.log(p) + (1 - y_) * np.log(1 - p))


BASE = ["logistica", "beto"]
CANDIDATAS = {"a (referencia)": BASE, **{f"a + {m}": BASE + [m] for m in NUEVOS}, "todos": BASE + NUEVOS}
subconjuntos = [list(c) for r in range(len(NUEVOS) + 1) for c in combinations(NUEVOS, r)]
elegidos_ll = {}
for k in range(1, 6):
    entrena = folds != k
    elegidos_ll[k] = min(subconjuntos, key=lambda s: log_loss(promedio(k, BASE + s)[entrena], y_np[entrena]))
    print(f"fold {k}: subconjunto elegido por log-loss = {elegidos_ll[k]}")

texto_solo = {}
for nombre, miembros in CANDIDATAS.items():
    p = np.empty(n)
    for k in range(1, 6):
        p[folds == k] = promedio(k, miembros)[folds == k]
    texto_solo[nombre] = {"log_loss": log_loss(p, y_np), "acc_total": acc_por_fold(p).mean(),
                          "acc_sin_hist": acc_por_fold(p, sin_historial).mean()}
p = np.empty(n)
for k in range(1, 6):
    p[folds == k] = promedio(k, BASE + elegidos_ll[k])[folds == k]
texto_solo["elegido (log-loss, 4 folds)"] = {"log_loss": log_loss(p, y_np), "acc_total": acc_por_fold(p).mean(),
                                             "acc_sin_hist": acc_por_fold(p, sin_historial).mean()}
pd.DataFrame(texto_solo).T.round(4)

fold 1: subconjunto elegido por log-loss = []
fold 2: subconjunto elegido por log-loss = []
fold 3: subconjunto elegido por log-loss = []
fold 4: subconjunto elegido por log-loss = []


fold 5: subconjunto elegido por log-loss = []


,log_loss,acc_total,acc_sin_hist
a (referencia),0.3293,0.8589,0.8036
a + nb_multinomial,0.3383,0.8513,0.7892
a + nb_bernoulli,0.3419,0.8493,0.7938
a + log_caracteres,0.3396,0.8563,0.8001
a + log_palabras,0.3378,0.8571,0.7998
a + svm_lineal,0.3366,0.8580,0.8023
todos,0.3481,0.8464,0.7854
"elegido (log-loss, 4 folds)",0.3293,0.8589,0.8036


## 3. Medición en el HistGB P7a y en el promedio final (contra la variante a)

**Qué hace `armar` / `oof`:** el pipeline de siempre con el transformer `texto` reemplazado por `passthrough` de la columna combinada (misma posición: `INDICES_CAT` no cambia). Para cada fold externo k la columna se arma con `columna(k)`.

**Control:** la candidata `a (referencia)` en el HistGB tiene que dar exactamente las OOF guardadas de la Fase T.

Además de cada candidata, la **elección anidada entre candidatas fijas**: para cada fold gana la de mejor accuracy en los otros 4 folds (a gana los empates) y se mide en el quinto.

In [5]:
def armar(modelo):
    pipeline = armar_pipeline_histgb_combos() if modelo == "histgb" else armar_pipeline_final()
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [nm for nm, _, _ in transformers].index("texto")
    transformers[i] = ("texto", "passthrough", ["t_texto"])
    return pipeline


def correr_fold(k, modelo, columna):
    entrena, valida = folds != k, folds == k
    X = train.assign(t_texto=columna)
    params = {"modelo__cat_features": INDICES_CAT} if modelo == "catboost" else {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipeline = armar(modelo).fit(X[entrena], y[entrena], **params)
    return k, pipeline.predict_proba(X[valida])[:, 1]


def oof(modelo, columna_de_fold):
    prob = np.empty(n)
    for k, p in Parallel(n_jobs=5)(delayed(correr_fold)(k, modelo, columna_de_fold(k)) for k in range(1, 6)):
        prob[folds == k] = p
    return prob


COLUMNAS_FOLD = {nombre: (lambda k, m=miembros: promedio(k, m)) for nombre, miembros in CANDIDATAS.items()}
COLUMNAS_FOLD["elegido (log-loss, 4 folds)"] = lambda k: promedio(k, BASE + elegidos_ll[k])

archivo_oof = os.path.join(carpeta_lab, "oof_texto_ensamble.csv")
guardado = pd.read_csv(archivo_oof) if os.path.exists(archivo_oof) else pd.DataFrame({"fold": folds, "y": y_np})
for nombre, columna_de_fold in COLUMNAS_FOLD.items():
    for modelo in ["histgb", "catboost"]:
        col = f"{modelo} | {nombre}"
        if col in guardado:
            continue
        inicio = time.time()
        guardado[col] = oof(modelo, columna_de_fold)
        guardado.to_csv(archivo_oof, index=False)
        print(f"{col:45s} {(time.time() - inicio) / 60:.1f} min", flush=True)

print("Control: máx |diferencia| de 'a (referencia)' con las OOF de la Fase T: HistGB",
      np.abs(guardado["histgb | a (referencia)"] - ref_hgb).max(),
      "| CatBoost", np.abs(guardado["catboost | a (referencia)"] - oof_t["catboost (a) promedio simple"]).max())

histgb | a (referencia)                       0.5 min


/Users/nicolasscravaglieri/Downloads/kaggle-ap-2026/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


catboost | a (referencia)                     1.6 min


histgb | a + nb_multinomial                   0.5 min


catboost | a + nb_multinomial                 1.7 min


histgb | a + nb_bernoulli                     0.6 min


... (8 outputs left out)

catboost | todos                              1.6 min


histgb | elegido (log-loss, 4 folds)          0.5 min


catboost | elegido (log-loss, 4 folds)        1.7 min


Control: máx |diferencia| de 'a (referencia)' con las OOF de la Fase T: HistGB 1.1102230246251565e-16 | CatBoost 1.1102230246251565e-16


In [6]:
resultados = {}
ref_sh_hgb, ref_sh_final = acc_por_fold(ref_hgb, sin_historial), acc_por_fold(ref_final, sin_historial)
probs_final = {}
for nombre in COLUMNAS_FOLD:
    p_h = guardado[f"histgb | {nombre}"].to_numpy()
    p_f = 0.5 * guardado[f"catboost | {nombre}"].to_numpy() + 0.5 * p_h
    probs_final[nombre] = p_f
    r_h, r_f = comparar(acc_por_fold(p_h), acc_ref_hgb), comparar(acc_por_fold(p_f), acc_ref_final)
    resultados[nombre] = {"histgb": r_h["media"], "histgb_dif": r_h["dif_media"], "histgb_folds": r_h["folds_gana"],
                          "histgb_pasa": r_h["pasa"], "histgb_sin_hist": acc_por_fold(p_h, sin_historial).mean(),
                          "final": r_f["media"], "final_dif": r_f["dif_media"], "final_desvio": r_f["dif_desvio"],
                          "final_folds": r_f["folds_gana"], "final_pasa": r_f["pasa"],
                          "final_sin_hist": acc_por_fold(p_f, sin_historial).mean()}


def eleccion_anidada(probs):
    nombres = list(CANDIDATAS)                       # a (referencia) primero: gana los empates
    acierto = {c: (probs[c] >= 0.5).astype(int) == y_np for c in nombres}
    final, elegidos = np.empty(n, bool), []
    for k in range(1, 6):
        mejor = max(nombres, key=lambda c: acierto[c][folds != k].mean())
        elegidos.append(mejor)
        final[folds == k] = acierto[mejor][folds == k]
    return (pd.Series(final).groupby(folds).mean().to_numpy(),
            pd.Series(final[sin_historial]).groupby(folds[sin_historial]).mean().to_numpy(), elegidos)


probs_hgb = {c: guardado[f"histgb | {c}"].to_numpy() for c in CANDIDATAS}
acc_h, sh_h, eleg_h = eleccion_anidada(probs_hgb)
acc_f, sh_f, eleg_f = eleccion_anidada({c: probs_final[c] for c in CANDIDATAS})
r_h, r_f = comparar(acc_h, acc_ref_hgb), comparar(acc_f, acc_ref_final)
resultados["ELECCIÓN ANIDADA entre candidatas"] = {
    "histgb": r_h["media"], "histgb_dif": r_h["dif_media"], "histgb_folds": r_h["folds_gana"], "histgb_pasa": r_h["pasa"],
    "histgb_sin_hist": sh_h.mean(), "final": r_f["media"], "final_dif": r_f["dif_media"], "final_desvio": r_f["dif_desvio"],
    "final_folds": r_f["folds_gana"], "final_pasa": r_f["pasa"], "final_sin_hist": sh_f.mean()}
print("Elección anidada — HistGB:", eleg_h)
print("Elección anidada — final: ", eleg_f)
print("Referencia (variante a): HistGB", acc_ref_hgb.mean().round(4), "| final", acc_ref_final.mean().round(4),
      "| sin historial: HistGB", ref_sh_hgb.mean().round(4), "final", ref_sh_final.mean().round(4))
tabla = pd.DataFrame(resultados).T
tabla.to_csv(os.path.join(carpeta_lab, "texto_ensamble.csv"))
tabla.round(5)

Elección anidada — HistGB: ['a (referencia)', 'a (referencia)', 'a (referencia)', 'a (referencia)', 'a (referencia)']
Elección anidada — final:  ['a (referencia)', 'a (referencia)', 'a (referencia)', 'a (referencia)', 'a (referencia)']
Referencia (variante a): HistGB 0.9425 | final 0.9435 | sin historial: HistGB 0.8948 final 0.8962


,histgb,histgb_dif,histgb_folds,histgb_pasa,histgb_sin_hist,final,final_dif,final_desvio,final_folds,final_pasa,final_sin_hist
a (referencia),0.942514,0.0,0,False,0.894836,0.943529,0.0,0.0,0,False,0.896197
a + nb_multinomial,0.941114,-0.0014,0,False,0.892248,0.942186,-0.001343,0.000883,0,False,0.893796
a + nb_bernoulli,0.941971,-0.000543,1,False,0.893165,0.942943,-0.000586,0.000576,1,False,0.894988
a + log_caracteres,0.942343,-0.000171,2,False,0.893485,0.942943,-0.000586,0.000846,1,False,0.895011
a + log_palabras,0.941871,-0.000643,1,False,0.892907,0.9425,-0.001029,0.000352,0,False,0.893592
a + svm_lineal,0.942071,-0.000443,1,False,0.893674,0.942843,-0.000686,0.000879,1,False,0.894413
todos,0.941386,-0.001129,1,False,0.891858,0.942043,-0.001486,0.000974,1,False,0.893646
"elegido (log-loss, 4 folds)",0.942514,0.0,0,False,0.894836,0.943529,0.0,0.0,0,False,0.896197
ELECCIÓN ANIDADA entre candidatas,0.942514,0.0,0,False,0.894836,0.943529,0.0,0.0,0,False,0.896197
